In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import patches as mpatches
import imageio

In [2]:
year = '2026'

In [3]:
pd.set_option('display.max_rows', 500)
plt.style.use('seaborn-v0_8-dark')

In [4]:
# data from grafana monitoring of lichess.org (websocket connections as proxy for players)
df = pd.read_csv(f'~/git/wclichess/{year}/wcdata{year}.csv')

In [5]:
# fixtures from https://fixturedownload.com/results/fifa-world-cup-2022
fixtures = pd.read_csv(f'{year}/fifa-world-cup-{year}-UTC.csv')

In [6]:
df = df.ffill() # cover missing points in grafana data
df['Time'] = pd.to_datetime(df.Time)
df['players'] = pd.to_numeric(df['players'].str.split().str[0]) # from 60.0 K string to 60.0 float

In [7]:
df['day'] = df.Time.dt.date
df['daytime'] = df.Time.dt.strftime('%H-%M')

In [8]:
# create table to see what matches were played when and by which teams
fixture_lookup = fixtures[['Home Team', 'Away Team']].stack().reset_index()\
    .merge(fixtures['Date'], how='inner', left_on='level_0', right_index=True)\
    .drop(columns=['level_0', 'level_1'])\
    .rename(columns={0: 'Country'})
fixture_lookup['Day'] = pd.to_datetime(fixture_lookup.Date, dayfirst=True).dt.date
fixture_lookup['Time'] = pd.to_datetime(fixture_lookup.Date, format="%d/%m/%Y %H:%M").dt.strftime('%H-%M')

In [9]:
days = fixture_lookup.Day.unique()
standard_day = df.day.unique()[1] if year == "2022" else df.day.unique()[2] # normal day example

In [10]:
# some names were too long to label neatly
fixture_lookup['Country'] = fixture_lookup.Country.replace({'Korea Republic': 'Korea', 'Saudi Arabia': 'S Arabia', 'Bosnia and Herzegovina': 'BIH'})

In [11]:
# getting the placement of the graph labels correct for each kickoff
# kickoffs = {
#     '10-00': ([40,40], [43, 105], 'k--'),
#     '13-00': ([52,52], [43, 105], 'k--'),
#     '15-00': ([60,60], [43, 105], 'k--'),
#     '16-00': ([64,64], [43, 105], 'k--'),
#     '19-00': ([76,76], [43, 105], 'k--'),
# }
# kickoff_labels = {
#     '10-00': 0.42,
#     '13-00': 0.52,
#     '15-00': 0.57,
#     '16-00': 0.60,
#     '19-00': 0.70
# }
kickoffs = {
    '00-00': ([0, 0], [20, 95], 'k--'),
    '00-30': ([2, 2], [20, 95], 'k--'),
    '01-00': ([4, 4], [20, 95], 'k--'),
    '01-30': ([6, 6], [20, 95], 'k--'),
    '02-00': ([8, 8], [20, 95], 'k--'),
    '03-00': ([12, 12], [20, 95], 'k--'),
    '04-00': ([16, 16], [20, 95], 'k--'),
    '10-00': ([40, 40], [20, 95], 'k--'),
    '13-00': ([52, 52], [20, 95], 'k--'),
    '15-00': ([60, 60], [20, 95], 'k--'),
    '16-00': ([64, 64], [20, 95], 'k--'),
    '17-00': ([68, 68], [20, 95], 'k--'),
    '18-00': ([72, 72], [20, 95], 'k--'),
    '19-00': ([76, 76], [20, 95], 'k--'),
    '20-00': ([80, 80], [20, 95], 'k--'),
    '20-30': ([82, 82], [20, 95], 'k--'),
    '21-00': ([84, 84], [20, 95], 'k--'),
    '22-00': ([88, 88], [20, 95], 'k--'),
    '23-00': ([92, 92], [20, 95], 'k--'),
    '23-30': ([94, 94], [20, 95], 'k--'),
}

# more granular data
if year == "2026":
    kickoffs = {
        time: ([x * 3 for x in x_coords], y_coords, style)
        for time, (x_coords, y_coords, style) in kickoffs.items()
    }

kickoff_labels = {
    '00-00': 0.14,
    '00-30': 0.16,
    '01-00': 0.17,
    '01-30': 0.19,
    '02-00': 0.20,
    '03-00': 0.22,
    '04-00': 0.25,
    '10-00': 0.42,
    '13-00': 0.52,
    '15-00': 0.56,
    '16-00': 0.59,
    '17-00': 0.62,
    '18-00': 0.65,
    '19-00': 0.68,
    '20-00': 0.71,
    '20-30': 0.73,
    '21-00': 0.75,
    '22-00': 0.77,
    '23-00': 0.80,
    '23-30': 0.82,
}

In [14]:
# make all the graphs for each day with both normal day and the specified day
for n, day in enumerate(days):
    f, ax = plt.subplots(figsize=(10, 7))
    plot_df = df.query('day == @day')[['daytime', 'players']]
    plot_df.plot(x='daytime', y='players', ax=ax, legend=False, color='g')
    plot_df = df.query('day == @standard_day')[['daytime', 'players']]
    plot_df.plot(x='daytime', y='players', ax=ax, legend=False, color='black')
    plt.title('Lichess.org Users During Football World Cup Relative to Normal Day', fontsize='18')
    # add labels for each kickoff
    for t in kickoffs.keys():
        matches = fixture_lookup.query('Day == @days[@n] & Time == @t')
        if not matches.empty:
            plt.plot(*kickoffs[t], lw=1, dashes=[2,2])
            plt.figtext(kickoff_labels[t], 0.15, matches.Country.to_string(index=False), fontsize='medium')
    plt.figtext(0.15, 0.7, f'Date: {day}', fontsize='xx-large')
    # credit
    plt.figtext(0.75, 0.85, 'github @michael1241', fontsize='small')
    plt.xlabel('Time (UTC)', fontsize='15')

    tick_hours = list(range(0, 25, 2))
    tick_positions = [h * 12 for h in tick_hours]
    tick_labels = [f"{h:02d}:00" for h in tick_hours]
    plt.xticks(tick_positions, tick_labels)
    
    plt.ylabel('Online Users in 1000s', fontsize='15')
    plt.ylim([30,110]) if year == "2022" else plt.ylim([0,130])
    normal_day_patch = mpatches.Patch(color='black', label='Normal traffic')
    wc_day_patch = mpatches.Patch(color='green', label='World Cup traffic')
    plt.legend(handles=[normal_day_patch, wc_day_patch], loc=2, prop={'size': 15})
    plt.savefig(f'./{year}/img/img_{n}.png', 
                transparent = False
               )
    plt.close()

In [15]:
frames = []
for n, day in enumerate(days):
    image = imageio.v2.imread(f'./{year}/img/img_{n}.png')
    frames.append(image)
# mkdir ./img/ first
imageio.mimsave(f'./{year}/img/output.gif', frames, fps = 0.4)